# From K-Means to GMM: Hard vs Soft Clustering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/unsupervised/k_means_clustering.ipynb)

This notebook implements K-Means clustering from scratch and shows how it relates to Gaussian Mixture Models (GMMs) through the EM algorithm.

**What you'll learn:**
- How K-Means works (Lloyd's algorithm)
- Why K-Means is a special case of EM with hard assignments
- How GMMs generalise K-Means with soft, probabilistic assignments
- When to use each approach

## 1. K-Means from Scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def kmeans(X, K, max_iter=100, seed=42):
    """
    K-Means clustering (Lloyd's algorithm).

    Args:
        X: (N, D) array of data points
        K: number of clusters
        max_iter: maximum iterations
        seed: random seed for reproducibility

    Returns:
        centroids: (K, D) final cluster centres
        labels: (N,) hard cluster assignments
        history: list of (centroids, labels) at each iteration
        distortions: list of objective function values
    """
    rng = np.random.default_rng(seed)
    N, D = X.shape

    # Step 1: Initialise centroids by picking K random data points
    indices = rng.choice(N, K, replace=False)
    centroids = X[indices].copy()

    history = []
    distortions = []

    for iteration in range(max_iter):
        # Step 2: ASSIGN — each point to nearest centroid
        distances = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)  # (N, K)
        labels = np.argmin(distances, axis=1)  # (N,)

        # Compute distortion (objective function)
        distortion = sum(np.sum((X[labels == k] - centroids[k])**2) for k in range(K))
        distortions.append(distortion)
        history.append((centroids.copy(), labels.copy()))

        # Step 3: UPDATE — move centroids to cluster means
        new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])

        # Check convergence
        if np.allclose(centroids, new_centroids):
            break

        centroids = new_centroids

    return centroids, labels, history, distortions

## 2. Generate Synthetic Data

Let's create three well-separated 2D clusters with different shapes.

In [ ]:
rng = np.random.default_rng(42)

# Three clusters with different shapes
n_per_cluster = 100
cluster_1 = rng.multivariate_normal([2, 2], [[0.8, 0.4], [0.4, 0.3]], n_per_cluster)
cluster_2 = rng.multivariate_normal([7, 7], [[0.5, -0.3], [-0.3, 0.8]], n_per_cluster)
cluster_3 = rng.multivariate_normal([2, 8], [[0.3, 0.0], [0.0, 0.6]], n_per_cluster)

X = np.vstack([cluster_1, cluster_2, cluster_3])
true_labels = np.array([0]*n_per_cluster + [1]*n_per_cluster + [2]*n_per_cluster)

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c='grey', alpha=0.5, s=20)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Unlabelled Data — Can We Find the Clusters?')
plt.grid(True, alpha=0.3)
plt.show()

## 3. Run K-Means

In [ ]:
centroids, labels, history, distortions = kmeans(X, K=3)

colours = ['#e74c3c', '#3498db', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Final clusters
ax = axes[0]
for k in range(3):
    mask = labels == k
    ax.scatter(X[mask, 0], X[mask, 1], c=colours[k], alpha=0.5, s=20, label=f'Cluster {k+1}')
ax.scatter(centroids[:, 0], centroids[:, 1], c='black', marker='X', s=200, zorder=5, label='Centroids')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('K-Means Result (K=3)')
ax.legend()
ax.grid(True, alpha=0.3)

# Distortion curve
ax = axes[1]
ax.plot(range(1, len(distortions) + 1), distortions, 'k-o', markersize=6)
ax.set_xlabel('Iteration')
ax.set_ylabel('Distortion $J$')
ax.set_title('K-Means Objective Decreases at Each Step')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Converged in {len(distortions)} iterations")
for k in range(3):
    n_points = (labels == k).sum()
    print(f"Cluster {k+1}: centroid = ({centroids[k, 0]:.2f}, {centroids[k, 1]:.2f}), {n_points} points")

## 4. Visualise the Algorithm Step by Step

Watch how K-Means alternates between assigning points and updating centroids.

In [ ]:
n_steps = min(6, len(history))
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for i, ax in enumerate(axes[:n_steps]):
    c, l = history[i]
    for k in range(3):
        mask = l == k
        ax.scatter(X[mask, 0], X[mask, 1], c=colours[k], alpha=0.4, s=15)
    ax.scatter(c[:, 0], c[:, 1], c='black', marker='X', s=150, zorder=5)
    ax.set_title(f'Iteration {i+1}')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(X[:, 0].min()-1, X[:, 0].max()+1)
    ax.set_ylim(X[:, 1].min()-1, X[:, 1].max()+1)

# Hide unused axes
for i in range(n_steps, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('K-Means: Assign → Update → Repeat', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. The Boundary Problem: Where K-Means Struggles

K-Means makes a hard decision for every point. But what about points near a boundary between two clusters? Let's highlight these uncertain points.

In [ ]:
# Compute distance ratios to find ambiguous points
distances = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)  # (N, K)
sorted_dists = np.sort(distances, axis=1)
ambiguity = 1 - (sorted_dists[:, 0] / sorted_dists[:, 1])  # higher = more ambiguous

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# K-Means: hard boundaries
ax = axes[0]
for k in range(3):
    mask = labels == k
    ax.scatter(X[mask, 0], X[mask, 1], c=colours[k], alpha=0.5, s=20)
ax.scatter(centroids[:, 0], centroids[:, 1], c='black', marker='X', s=200, zorder=5)
ax.set_title('K-Means: Hard Assignments')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.grid(True, alpha=0.3)

# Ambiguity: uncertain points
ax = axes[1]
sc = ax.scatter(X[:, 0], X[:, 1], c=ambiguity, cmap='RdYlGn_r', s=20, alpha=0.7)
ax.scatter(centroids[:, 0], centroids[:, 1], c='black', marker='X', s=200, zorder=5)
plt.colorbar(sc, ax=ax, label='Ambiguity')
ax.set_title('How Uncertain Is Each Assignment?')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. K-Means as EM with Hard Assignments

K-Means can be seen as a special case of the EM algorithm:
- **E-step**: Assign each point to the nearest centroid (hard 0/1 responsibilities)
- **M-step**: Recompute centroids as cluster means

Let's make this explicit by implementing K-Means in EM notation.

In [ ]:
def kmeans_as_em(X, K, max_iter=100, seed=42):
    """
    K-Means expressed as EM with hard (0/1) responsibilities.
    """
    rng = np.random.default_rng(seed)
    N, D = X.shape

    indices = rng.choice(N, K, replace=False)
    centroids = X[indices].copy()

    for iteration in range(max_iter):
        # E-STEP: Compute hard responsibilities r(n,k) ∈ {0, 1}
        distances = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
        r = np.zeros((N, K))
        r[np.arange(N), np.argmin(distances, axis=1)] = 1.0  # one-hot

        # M-STEP: Update centroids using responsibilities
        Nk = r.sum(axis=0)  # effective number of points per cluster
        new_centroids = (r.T @ X) / Nk[:, None]  # weighted mean

        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids

    return centroids, r

centroids_em, r_em = kmeans_as_em(X, K=3)
print("Responsibilities for the first 5 data points (hard, 0/1):")
print(r_em[:5])

## 7. GMM: Soft Responsibilities

Now let's compare with a GMM, which gives **probabilistic** responsibilities. A point near a boundary might be 60% cluster A and 40% cluster B.

In [ ]:
def gaussian_pdf_2d(X, mu, cov):
    """Multivariate Gaussian PDF."""
    D = len(mu)
    diff = X - mu
    cov_inv = np.linalg.inv(cov)
    det = np.linalg.det(cov)
    norm = 1.0 / (np.sqrt((2 * np.pi)**D * det))
    exponent = -0.5 * np.sum(diff @ cov_inv * diff, axis=1)
    return norm * np.exp(exponent)

def fit_gmm_2d(X, K=3, max_iter=100, seed=42):
    """
    Fit a 2D Gaussian Mixture Model using EM.
    """
    rng = np.random.default_rng(seed)
    N, D = X.shape

    # Initialise
    indices = rng.choice(N, K, replace=False)
    means = X[indices].copy()
    covariances = [np.eye(D) * np.var(X, axis=0) for _ in range(K)]
    weights = np.full(K, 1.0 / K)

    for iteration in range(max_iter):
        # E-STEP: Compute soft responsibilities
        resp = np.zeros((N, K))
        for k in range(K):
            resp[:, k] = weights[k] * gaussian_pdf_2d(X, means[k], covariances[k])
        resp /= resp.sum(axis=1, keepdims=True)

        # M-STEP: Update parameters
        Nk = resp.sum(axis=0)
        means = (resp.T @ X) / Nk[:, None]
        for k in range(K):
            diff = X - means[k]
            covariances[k] = (resp[:, k:k+1] * diff).T @ diff / Nk[k]
            covariances[k] += np.eye(D) * 1e-6  # regularisation
        weights = Nk / N

    return means, covariances, weights, resp

gmm_means, gmm_covs, gmm_weights, gmm_resp = fit_gmm_2d(X, K=3)

print("GMM responsibilities for the first 5 data points (soft, probabilities):")
print(np.round(gmm_resp[:5], 3))

## 8. Side-by-Side Comparison: Hard vs Soft Clustering

In [ ]:
def draw_ellipse(ax, mean, cov, colour, n_std=2):
    """Draw a covariance ellipse."""
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    width, height = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=mean, width=width, height=height, angle=angle,
                  edgecolor=colour, facecolor='none', linewidth=2, linestyle='--')
    ax.add_patch(ell)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# K-Means (hard assignments)
ax = axes[0]
for k in range(3):
    mask = labels == k
    ax.scatter(X[mask, 0], X[mask, 1], c=colours[k], alpha=0.5, s=20)
ax.scatter(centroids[:, 0], centroids[:, 1], c='black', marker='X', s=200, zorder=5)
ax.set_title('K-Means: Hard Assignments\n(each point → exactly one cluster)', fontsize=11)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.grid(True, alpha=0.3)

# GMM (soft assignments with covariance ellipses)
ax = axes[1]
gmm_labels = np.argmax(gmm_resp, axis=1)
for k in range(3):
    mask = gmm_labels == k
    # Use max responsibility as alpha to show uncertainty
    alphas = gmm_resp[mask, k]
    ax.scatter(X[mask, 0], X[mask, 1], c=colours[k], alpha=0.3 + 0.5*alphas, s=20)
    draw_ellipse(ax, gmm_means[k], gmm_covs[k], colours[k])
ax.scatter(gmm_means[:, 0], gmm_means[:, 1], c='black', marker='X', s=200, zorder=5)
ax.set_title('GMM: Soft Assignments\n(each point → probability per cluster)', fontsize=11)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Choosing K: The Elbow Method

K-Means requires you to specify the number of clusters. The elbow method plots distortion vs K.

In [ ]:
K_range = range(1, 8)
final_distortions = []

for K in K_range:
    _, _, _, dists = kmeans(X, K=K)
    final_distortions.append(dists[-1])

plt.figure(figsize=(7, 4))
plt.plot(list(K_range), final_distortions, 'ko-', markersize=8)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.5, label='True K=3')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Final Distortion $J$')
plt.title('Elbow Method for Choosing K')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. The Unified View: K-Means → GMM → HMM

K-Means, GMMs, and HMMs are all variations on the same theme: **assign data points to hidden groups and estimate group parameters**. The key differences are in their assumptions:

| | K-Means | GMM | HMM |
|---|---------|-----|-----|
| **Responsibilities** | Hard (0 or 1) | Soft (probabilities) | Soft (probabilities) |
| **Data assumption** | i.i.d. | i.i.d. | Sequentially dependent |
| **Cluster shape** | Spherical (isotropic) | Elliptical (full covariance) | Elliptical (full covariance) |
| **Objective** | Distortion (distance) | Log-likelihood | Log-likelihood |
| **Fitting algorithm** | Lloyd's (EM, hard) | EM (soft) | Baum-Welch (EM for sequences) |

## Exercises

1. **Initialisation sensitivity** — Run K-Means with 10 different random seeds. How often does it find the "right" clusters? Try K-Means++ initialisation.

2. **Non-spherical clusters** — Create elongated clusters (high covariance in one direction). How does K-Means handle them vs GMM?

3. **Old Faithful** — Apply both K-Means and GMM to the Old Faithful eruption data (2D: duration + waiting time). Compare the cluster boundaries.

4. **Soft K-Means** — Modify the K-Means code to use soft assignments: `r_nk = exp(-β * d_nk²) / Σ exp(-β * d_nj²)`. What happens as β → ∞?

5. **Comparison with scikit-learn** — Verify your implementation matches `sklearn.cluster.KMeans` on the same data.